# Music Annotation Project V9.0A
## Audit-Resolved Mask-Authority Transfer Experiment, Streaming Fix

**Runtime:** CPU when Notebook 06 confidence maps already exist.

This is a clean replacement for the mixed resolver versions. It contains:

- one resolver only;
- no `paired`, `resolved`, `draft_resolution`, or `review_resolution` legacy variables;
- Notebook 07 `PageXXX` as the canonical page identity;
- backward resolution from Notebook 07 review records to Notebook 06 draft artefacts;
- explicit unresolved-lineage policy and diagnostics;
- separate raw-YOLO, confidence-gated-YOLO and reviewed-mask arms;
- clean truth loaded only after masks and corrected outputs are locked;
- a curated Final Upload Package.

### Scientific boundary

The YOLO confidence map, authoritative annotation mask, prepared removal mask and reconstruction routing are separate artefacts. Detector confidence is not correction safety. The notebook does not authorise automatic mask authority or production output.

### Gate policy

The notebook deliberately leaves the Notebook 06 detector-confidence gate unset. The retrieved project record does not establish a frozen detector-confidence threshold, and the V8.2 0.90 value is an observed-print probability threshold, not a YOLO mask-authority threshold. Set the two clearly labelled configuration values only when the genuine historical detector policy is identified.

In [1]:
# PROVENANCE: standard project Colab bootstrap.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib pyarrow

Mounted at /content/drive


## 1. Configuration and run identity

In [2]:
# PROVENANCE: project paths from Notebooks 06, 07 and 09; V9.0A-specific output layout.
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, re, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR = ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR = ROOT/'human_in_the_middle'/'audit'
CLEAN_DIR = ROOT/'source'/'clean'
OUTPUT_ROOT = ROOT/'mask_authority_transfer_v9_0a'

# Optional restriction using Notebook 07 PageXXX identities.
PAGE_ALLOWLIST = None

# Lineage policy:
# - exclude_and_report: continue with resolvable reviewed pages and report every exclusion.
# - require_complete: stop unless every Notebook 07 review audit resolves.
UNRESOLVED_LINEAGE_POLICY = 'exclude_and_report'

# Established Notebook 06 detector-confidence mask gate.
# Do not substitute V8.2 observed-print probabilities here.
EXPERIMENTAL_YOLO_GATE = 0.90
EXPERIMENTAL_YOLO_GATE_POLICY_ID = 'V9.0A_EXPLORATORY_YOLO_CONFIDENCE_GE_090'
# This is an experimental Notebook 06 confidence-map cut-off, not production authority.

# Frozen canonical correction contract from the V7.8/Notebook 09 lineage.
MASK_DILATION = 2
CROP_MARGIN = 48
GEOMETRY_THRESHOLD = 80
BAND_WIDTH = 4

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v9_0a_audit_resolved'
OUT = OUTPUT_ROOT/'runs'/RUN_ID
INPUTS=OUT/'01_inputs'; MASKS=OUT/'02_mask_arms'; OUTPUTS=OUT/'03_corrected_pages'
LOCKS=OUT/'04_locks'; SCORE=OUT/'05_scoring'; VISUAL=OUT/'06_visual_audit'
AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,OUTPUTS,LOCKS,SCORE,VISUAL,AUDIT,UPLOAD]:
    p.mkdir(parents=True,exist_ok=False)

ARMS=['reviewed_comparator','raw_yolo','confidence_gated_yolo','reviewer_corrected_yolo']
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
print('Output:',OUT)

Output: /content/drive/MyDrive/MusicAnnotationProject/mask_authority_transfer_v9_0a/runs/20261110_151505_v9_0a_audit_resolved


## 2. Integrity helpers

In [3]:
# PROVENANCE: consolidated project integrity helpers from the V7.8/Notebook 09 lineage.
def require(condition,message):
    if not bool(condition):
        try:
            (AUDIT/'fatal_failure.json').write_text(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'message':str(message)},indent=2))
        except Exception:
            pass
        raise RuntimeError(message)

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def sha256_file(path):
    path=Path(path); require(path.exists() and path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def read_gray(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(im is not None,f'Unreadable image: {path}'); return im

def binary(mask):
    return np.where(mask>0,255,0).astype(np.uint8)

def normalised_path_string(value):
    if value is None or not str(value).strip(): return None
    return str(value).strip().replace('\\','/').rstrip('/').casefold()

## 3. Load Notebook 06 and Notebook 07 audit records

In [4]:
# PROVENANCE: adapted from Notebook 09 read_audit_records.
def load_audit_records(folder,audit_type):
    records=[]; failures=[]
    for audit_path in sorted(Path(folder).glob('*.json')):
        try:
            record=json.loads(audit_path.read_text())
            if not isinstance(record,dict): raise ValueError('Audit root must be a JSON object.')
            record['_audit_path']=str(audit_path)
            record['_audit_name']=audit_path.name
            record['_audit_stem']=audit_path.stem
            record['_audit_type']=audit_type
            record['_audit_mtime_ns']=audit_path.stat().st_mtime_ns
            records.append(record)
        except Exception as error:
            failures.append({'audit_type':audit_type,'audit_path':str(audit_path),'error':str(error)})
    return records,failures

def recorded_path(record,possible_fields):
    for field in possible_fields:
        value=record.get(field)
        if value is not None and str(value).strip(): return Path(str(value)),field
    return None,None

def required_existing_path(record,possible_fields,audit_type):
    attempted=[]
    for field in possible_fields:
        value=record.get(field)
        if value is None or not str(value).strip(): continue
        path=Path(str(value)); attempted.append({'field':field,'path':str(path),'exists':path.exists()})
        if path.exists(): return path,field
    raise RuntimeError(f'{audit_type} audit has no existing file for fields {possible_fields}: {record.get("_audit_path")}. Attempted: {attempted}')

def audit_source_path(record,audit_type):
    fields=['source_file','source','input_file','input_path'] if audit_type=='draft' else ['source','source_file','original_source','input_file','input_path']
    return recorded_path(record,fields)

def safe_recorded_md5(record):
    value=record.get('source_md5')
    if value is None:return None
    value=str(value).strip().lower()
    return value or None

def stable_draft_signature(record):
    fields=['source_file','source_md5','draft_mask','confidence_map','model_path','model_md5','prediction_confidence','shape']
    return object_hash({field:record.get(field) for field in fields})

def collapse_identical_drafts(records,context):
    unique={r['_audit_path']:r for r in records}; records=list(unique.values())
    if len(records)==1:return records[0],'unique',1
    groups={}
    for record in records: groups.setdefault(stable_draft_signature(record),[]).append(record)
    if len(groups)!=1:
        raise RuntimeError(f'{context}: conflicting Notebook 06 draft audits: {[r["_audit_path"] for r in records]}')
    replicas=next(iter(groups.values()))
    return sorted(replicas,key=lambda r:r['_audit_path'])[0],'identical_replicas_collapsed',len(replicas)

DRAFT_AUDITS,draft_failures=load_audit_records(DRAFT_AUDIT_DIR,'draft')
REVIEW_AUDITS,review_failures=load_audit_records(REVIEW_AUDIT_DIR,'review')
failures=draft_failures+review_failures
if failures:
    pd.DataFrame(failures).to_csv(AUDIT/'audit_parse_failures.csv',index=False)
    raise RuntimeError(f'{len(failures)} HITL audit records could not be read.')
require(DRAFT_AUDITS,f'No Notebook 06 draft audits found in {DRAFT_AUDIT_DIR}')
require(REVIEW_AUDITS,f'No Notebook 07 review audits found in {REVIEW_AUDIT_DIR}')
print('Notebook 06 draft audits:',len(DRAFT_AUDITS))
print('Notebook 07 review audits:',len(REVIEW_AUDITS))

Notebook 06 draft audits: 173
Notebook 07 review audits: 173


## 4. Resolve Notebook 07 PageXXX records backwards to Notebook 06 artefacts

In [5]:
# PROVENANCE: NEW V9.0A resolver using Notebook 09 audit fields.
# This is the sole resolver. There is no whole-directory `paired` source_md5 intersection.
drafts_by_recorded_path={}; drafts_by_actual_md5={}; drafts_by_stored_md5={}; drafts_by_source_name={}; draft_inventory=[]
for draft in DRAFT_AUDITS:
    source_path,source_field=audit_source_path(draft,'draft')
    path_key=normalised_path_string(source_path)
    actual_md5=md5_file(source_path).lower() if source_path is not None and source_path.exists() else None
    stored_md5=safe_recorded_md5(draft)
    source_name=source_path.name.casefold() if source_path is not None else None
    if path_key:drafts_by_recorded_path.setdefault(path_key,[]).append(draft)
    if actual_md5:drafts_by_actual_md5.setdefault(actual_md5,[]).append(draft)
    if stored_md5:drafts_by_stored_md5.setdefault(stored_md5,[]).append(draft)
    if source_name:drafts_by_source_name.setdefault(source_name,[]).append(draft)
    draft_inventory.append({'draft_audit_path':draft['_audit_path'],'recorded_source_field':source_field,'recorded_source_path':str(source_path) if source_path else None,'recorded_source_exists':bool(source_path and source_path.exists()),'actual_source_md5':actual_md5,'stored_source_md5':stored_md5,'source_filename':source_path.name if source_path else None,'draft_mask':draft.get('draft_mask'),'confidence_map':draft.get('confidence_map'),'model_path':draft.get('model_path'),'model_md5':draft.get('model_md5'),'prediction_confidence':draft.get('prediction_confidence')})
pd.DataFrame(draft_inventory).to_csv(AUDIT/'notebook06_draft_audit_inventory.csv',index=False)

resolved_rows=[]; attempt_rows=[]; unresolved_rows=[]
for review in REVIEW_AUDITS:
    page=str(review.get('page','')).strip() or Path(review['_audit_path']).stem
    if not re.fullmatch(r'Page\d+',page,flags=re.IGNORECASE):
        unresolved_rows.append({'page':page,'review_audit_path':review['_audit_path'],'reason':'invalid_notebook07_page_identity'}); continue
    review_source_path,review_source_field=audit_source_path(review,'review')
    review_path_key=normalised_path_string(review_source_path)
    review_stored_md5=safe_recorded_md5(review)
    review_actual_md5=md5_file(review_source_path).lower() if review_source_path is not None and review_source_path.exists() else None
    review_source_name=review_source_path.name.casefold() if review_source_path is not None else None
    candidate_sets=[]
    def add_candidates(method,strength,candidates):
        if candidates:candidate_sets.append({'method':method,'strength':strength,'candidates':candidates})
    add_candidates('exact_recorded_source_path',1,drafts_by_recorded_path.get(review_path_key,[]) if review_path_key else [])
    add_candidates('actual_source_file_md5',2,drafts_by_actual_md5.get(review_actual_md5,[]) if review_actual_md5 else [])
    add_candidates('review_actual_md5_to_draft_stored_md5',3,drafts_by_stored_md5.get(review_actual_md5,[]) if review_actual_md5 else [])
    add_candidates('review_stored_md5_to_draft_actual_md5',4,drafts_by_actual_md5.get(review_stored_md5,[]) if review_stored_md5 else [])
    add_candidates('stored_source_md5',5,drafts_by_stored_md5.get(review_stored_md5,[]) if review_stored_md5 else [])
    add_candidates('unique_original_source_filename',6,drafts_by_source_name.get(review_source_name,[]) if review_source_name else [])
    for cs in candidate_sets:
        for candidate in cs['candidates']:
            attempt_rows.append({'page':page,'review_audit_path':review['_audit_path'],'review_source_field':review_source_field,'review_source_path':str(review_source_path) if review_source_path else None,'review_source_exists':bool(review_source_path and review_source_path.exists()),'review_actual_md5':review_actual_md5,'review_stored_md5':review_stored_md5,'resolution_method':cs['method'],'resolution_strength':cs['strength'],'draft_audit_candidate':candidate['_audit_path']})
    if not candidate_sets:
        unresolved_rows.append({'page':page,'review_audit_path':review['_audit_path'],'review_source_path':str(review_source_path) if review_source_path else None,'review_actual_md5':review_actual_md5,'review_stored_md5':review_stored_md5,'reason':'no_notebook06_candidate_found'}); continue
    chosen=None; chosen_method=None; replica_status=None; replica_count=None; errors=[]
    for cs in sorted(candidate_sets,key=lambda x:x['strength']):
        try:
            chosen,replica_status,replica_count=collapse_identical_drafts(cs['candidates'],f'{page} via {cs["method"]}')
            chosen_method=cs['method']; break
        except RuntimeError as error:
            errors.append({'method':cs['method'],'error':str(error)})
    if chosen is None:
        unresolved_rows.append({'page':page,'review_audit_path':review['_audit_path'],'reason':'all_candidate_methods_ambiguous','details':json.dumps(errors,sort_keys=True)}); continue
    try:
        observed_path,observed_field=required_existing_path(review,['aligned_output','aligned_page','observed_output'],'Notebook 07 review')
        reviewed_mask_path,reviewed_mask_field=required_existing_path(review,['corrected_mask_output','corrected_mask','mask_output'],'Notebook 07 review')
        raw_mask_path,raw_mask_field=required_existing_path(chosen,['draft_mask','draft_mask_path'],'Notebook 06 draft')
        confidence_path,confidence_field=required_existing_path(chosen,['confidence_map','confidence_map_path'],'Notebook 06 draft')
        draft_source_path,draft_source_field=required_existing_path(chosen,['source_file','source','input_file','input_path'],'Notebook 06 draft')
        observed=read_gray(observed_path); raw=read_gray(raw_mask_path); reviewed=read_gray(reviewed_mask_path); confidence=np.load(confidence_path).astype(np.float32)
        require(observed.shape==raw.shape==reviewed.shape==confidence.shape,f'{page}: shape mismatch observed={observed.shape}, raw={raw.shape}, reviewed={reviewed.shape}, confidence={confidence.shape}')
        require(np.isfinite(confidence).all(),f'{page}: confidence map contains non-finite values')
        draft_actual_md5=md5_file(draft_source_path).lower(); draft_stored_md5=safe_recorded_md5(chosen)
        if draft_stored_md5 is not None:require(draft_actual_md5==draft_stored_md5,f'{page}: selected Notebook 06 audit has stale source_md5')
        if review_actual_md5 is not None:require(draft_actual_md5==review_actual_md5,f'{page}: selected Notebook 06 and Notebook 07 original sources differ')
        resolved_rows.append({'page':page,'source_md5':draft_actual_md5,'original_source_path':str(draft_source_path),'observed_path':str(observed_path),'raw_yolo_mask_path':str(raw_mask_path),'confidence_map_path':str(confidence_path),'reviewed_mask_path':str(reviewed_mask_path),'draft_audit_path':chosen['_audit_path'],'review_audit_path':review['_audit_path'],'resolution_method':chosen_method,'draft_replica_status':replica_status,'draft_replica_count':replica_count,'draft_model_path':chosen.get('model_path'),'draft_model_md5':chosen.get('model_md5'),'draft_prediction_confidence':chosen.get('prediction_confidence')})
    except Exception as error:
        unresolved_rows.append({'page':page,'review_audit_path':review['_audit_path'],'draft_audit_path':chosen['_audit_path'],'resolution_method':chosen_method,'reason':'resolved_candidate_failed_validation','details':str(error)})

pd.DataFrame(attempt_rows).to_csv(AUDIT/'notebook07_to_notebook06_resolution_attempts.csv',index=False)
unresolved=pd.DataFrame(unresolved_rows,columns=['page','review_audit_path','draft_audit_path','review_source_path','review_actual_md5','review_stored_md5','resolution_method','reason','details'])
unresolved.to_csv(AUDIT/'unresolved_review_lineages.csv',index=False)
resolution=pd.DataFrame(resolved_rows)
require(not resolution.empty,f'No Notebook 07 reviewed pages resolved to valid Notebook 06 artefacts. Inspect {AUDIT}.')
require('page' in resolution.columns,f'Resolution table has no page column. Columns={resolution.columns.tolist()}')
require(not resolution.page.duplicated().any(),'Duplicate canonical PageXXX identities remain after resolution.')
require(not resolution.source_md5.duplicated().any(),'More than one PageXXX resolves to the same original Notebook 06 source.')
if PAGE_ALLOWLIST is not None:
    requested=set(map(str,PAGE_ALLOWLIST)); missing=sorted(requested-set(resolution.page.astype(str))); require(not missing,f'Allowlisted pages lack complete lineage: {missing}'); resolution=resolution[resolution.page.astype(str).isin(requested)].copy()
resolution=resolution.sort_values('page').reset_index(drop=True)
if UNRESOLVED_LINEAGE_POLICY=='require_complete':require(len(unresolved)==0,f'{len(unresolved)} review records are unresolved.')
require(UNRESOLVED_LINEAGE_POLICY in {'exclude_and_report','require_complete'},'Unsupported unresolved-lineage policy.')
resolution.to_csv(AUDIT/'hitl_input_resolution.csv',index=False)
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook06_draft_audits':len(DRAFT_AUDITS),'notebook07_review_audits':len(REVIEW_AUDITS),'resolved_pages':len(resolution),'unresolved_review_records':len(unresolved),'unresolved_lineage_policy':UNRESOLVED_LINEAGE_POLICY,'resolution_methods':resolution.resolution_method.value_counts().sort_index().to_dict(),'canonical_page_identity':'Notebook 07 page / PageXXX','cross_stage_filename_stem_join':False,'whole_directory_md5_intersection_required':False}
atomic_json(AUDIT/'hitl_resolution_summary.json',summary)
EXPECTED=['page','source_md5','resolution_method','draft_replica_status','draft_replica_count','raw_yolo_mask_path','confidence_map_path','reviewed_mask_path','draft_audit_path','review_audit_path']
missing=[x for x in EXPECTED if x not in resolution.columns]; require(not missing,f'Resolution table missing {missing}. Actual={resolution.columns.tolist()}')
print(json.dumps(summary,indent=2)); display(resolution[EXPECTED])

{
  "created_utc": "2026-11-10T15:21:24.891190+00:00",
  "notebook06_draft_audits": 173,
  "notebook07_review_audits": 173,
  "resolved_pages": 151,
  "unresolved_review_records": 22,
  "unresolved_lineage_policy": "exclude_and_report",
  "resolution_methods": {
    "exact_recorded_source_path": 151
  },
  "canonical_page_identity": "Notebook 07 page / PageXXX",
  "cross_stage_filename_stem_join": false,
  "whole_directory_md5_intersection_required": false
}


,page,source_md5,resolution_method,draft_replica_status,draft_replica_count,raw_yolo_mask_path,confidence_map_path,reviewed_mask_path,draft_audit_path,review_audit_path
0,Page086,64834dba59a52b372f94346e406ad8e2,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
1,Page088,49e5da2d71739ee373491355d54f836f,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
2,Page089,90ac5ddd13a5b926a4bdec472de7f268,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
3,Page090,e8721d708890bd71354ac0c5449fac22,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
4,Page091,835984ae7a56f77d3668a9c7aa28e3ce,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
...,...,...,...,...,...,...,...,...,...,...
146,Page257,018092b34c2d7ec600b13efd631bc1ec,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
147,Page258,d3d1b779e5ebae06478ecdca4e266d9b,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
148,Page259,990d1ca9672f2f30c338b32284160ac5,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...
149,Page260,e69142d3925d60092e3754e54b71e981,exact_recorded_source_path,unique,1,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...,/content/drive/MyDrive/MusicAnnotationProject/...


## 5. Validate the exploratory Notebook 06 detector-confidence gate


In [6]:
# PROVENANCE: V9.0A experiment control.
# Notebook 06 raw generation threshold and the exploratory higher gate remain distinct.
require(EXPERIMENTAL_YOLO_GATE is not None,'Set EXPERIMENTAL_YOLO_GATE.')
require(EXPERIMENTAL_YOLO_GATE_POLICY_ID and str(EXPERIMENTAL_YOLO_GATE_POLICY_ID).strip(),'Set the gate experiment identity.')
EXPERIMENTAL_YOLO_GATE=float(EXPERIMENTAL_YOLO_GATE)
require(0.0<=EXPERIMENTAL_YOLO_GATE<=1.0,'Gate must be in [0,1].')
raw_values=pd.to_numeric(resolution.draft_prediction_confidence,errors='coerce').dropna().unique().tolist()
require(len(raw_values)<=1,f'Notebook 06 audits disagree on prediction_confidence: {raw_values}')
raw_threshold=float(raw_values[0]) if raw_values else None
require(raw_threshold is None or EXPERIMENTAL_YOLO_GATE>raw_threshold,'The exploratory gate must exceed the raw Notebook 06 draft threshold, otherwise the two arms collapse.')
gate_record={
    'gate':EXPERIMENTAL_YOLO_GATE,
    'policy_id':str(EXPERIMENTAL_YOLO_GATE_POLICY_ID),
    'raw_notebook06_prediction_confidence':raw_threshold,
    'optimised_in_v9_0a':False,
    'production_authority':False,
    'v8_2_observed_print_policy_used':False,
}
atomic_json(AUDIT/'yolo_gate_identity.json',gate_record)
print(json.dumps(gate_record,indent=2))

{
  "gate": 0.9,
  "policy_id": "V9.0A_EXPLORATORY_YOLO_CONFIDENCE_GE_090",
  "raw_notebook06_prediction_confidence": 0.2,
  "optimised_in_v9_0a": false,
  "production_authority": false,
  "v8_2_observed_print_policy_used": false
}


## 6. Canonical correction functions

In [7]:
# PROVENANCE: TRANSPLANTED from the canonical V7.8 block used by Notebook 09.
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]; require(np.array_equal(out[mask==0],original[mask==0]),'Outside-mask drift'); return out

def prepare_mask(seed,dilation=MASK_DILATION):
    mask=binary(seed)
    if dilation>0: mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return mask

def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8); h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=8: boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0): ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def telea_reconstruct(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]; cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA); local=result[y0:y1,x0:x1]; local[mk>0]=cand[mk>0]
    return immutable(observed,result,mask)

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32); num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma); fallback=cv2.GaussianBlur(observed,(0,0),sigma); return np.clip(np.where(den>0.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def support_maps(observed,mask):
    dark=(observed<190)&(mask==0); h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0; v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0; n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8); return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask); normalised=cv2.divide(observed,np.maximum(paper,1),scale=255); horizontal,vertical,compact=support_maps(observed,mask); m=mask>0; preserve=m&(horizontal|vertical|compact)&(observed<200); remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD); ambiguous=m&~preserve&~remove; result=observed.copy(); result[remove]=paper[remove]; result[ambiguous]=telea[ambiguous]; result[preserve]=observed[preserve]; return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact}

## 7. Execute paired mask-authority arms

In [8]:
# PROVENANCE: V9.0A bounded-memory paired orchestration.
# Full-page arrays are persisted and released after each arm/page.
import gc
runtime={}; mask_rows=[]; output_rows=[]; edit_rows=[]
for _,r in resolution.iterrows():
    page=str(r.page)
    observed=read_gray(r.observed_path)
    raw=binary(read_gray(r.raw_yolo_mask_path))
    reviewed=binary(read_gray(r.reviewed_mask_path))
    confidence=np.load(r.confidence_map_path).astype(np.float32)
    gated=np.where((raw>0)&(confidence>=EXPERIMENTAL_YOLO_GATE),255,0).astype(np.uint8)
    require(observed.shape==raw.shape==reviewed.shape==confidence.shape==gated.shape,f'{page}: execution shape mismatch')
    arms={'reviewed_comparator':reviewed,'raw_yolo':raw,'confidence_gated_yolo':gated,'reviewer_corrected_yolo':reviewed.copy()}
    runtime[page]={}
    edit_rows.append({'page':page,'raw_pixels':int((raw>0).sum()),'reviewed_pixels':int((reviewed>0).sum()),'reviewer_additions':int(((reviewed>0)&(raw==0)).sum()),'reviewer_erasures':int(((reviewed==0)&(raw>0)).sum()),'raw_reviewed_xor':int(np.logical_xor(raw>0,reviewed>0).sum())})
    for arm,seed in arms.items():
        prepared=prepare_mask(seed)
        telea=telea_reconstruct(observed,prepared)
        corrected,routes=geometry_t80(observed,prepared,telea)
        require(np.array_equal(corrected[prepared==0],observed[prepared==0]),f'{page}/{arm}: outside-mask drift')
        mask_dir=MASKS/arm; mask_dir.mkdir(parents=True,exist_ok=True)
        seed_path=mask_dir/f'{page}.png'; require(cv2.imwrite(str(seed_path),seed),f'{page}/{arm}: failed to write seed')
        output_dir=OUTPUTS/arm/page; output_dir.mkdir(parents=True,exist_ok=False)
        arrays=[('observed',observed),('authoritative_seed',seed),('prepared_mask',prepared),('telea',telea),('corrected',corrected),('route_preserve',routes['preserve'].astype(np.uint8)*255),('route_remove',routes['remove'].astype(np.uint8)*255),('route_ambiguous',routes['ambiguous'].astype(np.uint8)*255)]
        for name,array in arrays: require(cv2.imwrite(str(output_dir/f'{name}.png'),array),f'{page}/{arm}: failed to write {name}')
        runtime[page][arm]={'seed_path':str(seed_path),'output_dir':str(output_dir)}
        mask_rows.append({'page':page,'arm':arm,'seed_pixels':int((seed>0).sum()),'prepared_mask_pixels':int((prepared>0).sum()),'seed_sha256':array_hash(seed),'prepared_mask_sha256':array_hash(prepared),'source_md5':r.source_md5})
        output_rows.append({'page':page,'arm':arm,'corrected_sha256':array_hash(corrected),'outside_mask_drift':0,'changed_pixels':int((corrected!=observed).sum())})
        del prepared,telea,corrected,routes,arrays
        gc.collect()
    del observed,raw,reviewed,confidence,gated,arms
    gc.collect()
mask_manifest=pd.DataFrame(mask_rows); output_manifest=pd.DataFrame(output_rows); reviewer_edits=pd.DataFrame(edit_rows)
mask_manifest.to_csv(LOCKS/'mask_arm_manifest.csv',index=False)
output_manifest.to_csv(LOCKS/'corrected_output_manifest.csv',index=False)
reviewer_edits.to_csv(SCORE/'reviewer_edits.csv',index=False)
reviewed_hashes=(mask_manifest[mask_manifest.arm.isin(['reviewed_comparator','reviewer_corrected_yolo'])].pivot(index='page',columns='arm',values='seed_sha256'))
require(set(reviewed_hashes.columns)=={'reviewed_comparator','reviewer_corrected_yolo'},'Reviewed-role mask records are incomplete.')
require(reviewed_hashes.notna().all().all(),'One or more reviewed-role seed hashes are missing.')
require((reviewed_hashes.reviewed_comparator==reviewed_hashes.reviewer_corrected_yolo).all(),'Reviewed-role seed masks unexpectedly differ.')
print('Pages:',len(runtime),'arm executions:',len(output_manifest))

Pages: 151 arm executions: 604


## 8. Prediction and output lock

In [9]:
# PROVENANCE: project truth-firewall pattern.
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'pages':sorted(runtime),'source_md5_roster':resolution[['page','source_md5']].sort_values('page').to_dict('records'),'hitl_resolution_sha256':sha256_file(AUDIT/'hitl_input_resolution.csv'),'resolution_attempts_sha256':sha256_file(AUDIT/'notebook07_to_notebook06_resolution_attempts.csv'),'unresolved_lineages_sha256':sha256_file(AUDIT/'unresolved_review_lineages.csv'),'resolution_summary_sha256':sha256_file(AUDIT/'hitl_resolution_summary.json'),'yolo_gate_identity_sha256':sha256_file(AUDIT/'yolo_gate_identity.json'),'mask_manifest_sha256':sha256_file(LOCKS/'mask_arm_manifest.csv'),'output_manifest_sha256':sha256_file(LOCKS/'corrected_output_manifest.csv'),'truth_loaded':False}
lock['lock_id']=object_hash(lock); atomic_json(LOCKS/'prediction_and_output_lock.json',lock)
print('Lock ID:',lock['lock_id'])

Lock ID: c5d7cc2e10ec751bfd9bd3e7708d4f65cbf83ebdef63f7230654c46f9c51348a


## 9. Post-lock paired comparisons

In [10]:
# PROVENANCE: V9.0A disk-backed paired comparison.
import gc
def mask_metrics(candidate,reference):
    c=candidate>0; r=reference>0; inter=int((c&r).sum()); union=int((c|r).sum())
    return {'iou':inter/max(union,1),'false_positive_pixels_vs_reviewed':int((c&~r).sum()),'false_negative_pixels_vs_reviewed':int((~c&r).sum()),'xor_pixels':int((c^r).sum())}
rows=[]
for page in sorted(runtime):
    ref_mask=binary(read_gray(MASKS/'reviewed_comparator'/f'{page}.png'))
    ref_out=read_gray(OUTPUTS/'reviewed_comparator'/page/'corrected.png')
    for arm in ARMS:
        candidate_mask=binary(read_gray(MASKS/arm/f'{page}.png'))
        candidate_out=read_gray(OUTPUTS/arm/page/'corrected.png')
        require(candidate_mask.shape==ref_mask.shape,f'{page}/{arm}: mask shape mismatch')
        require(candidate_out.shape==ref_out.shape,f'{page}/{arm}: corrected-output shape mismatch')
        q=mask_metrics(candidate_mask,ref_mask)
        q.update({'page':page,'arm':arm,'corrected_xor_pixels_vs_reviewed':int((candidate_out!=ref_out).sum()),'corrected_mae_vs_reviewed':float(np.abs(candidate_out.astype(np.int16)-ref_out.astype(np.int16)).mean())})
        rows.append(q)
        del candidate_mask,candidate_out
    del ref_mask,ref_out
    gc.collect()
paired=pd.DataFrame(rows)
require(len(paired)==len(runtime)*len(ARMS),f'Paired table incomplete: expected {len(runtime)*len(ARMS)}, found {len(paired)}')
paired.to_csv(SCORE/'paired_mask_and_output_evidence.csv',index=False)
summary_df=paired.groupby('arm',as_index=False).agg(pages=('page','nunique'),mean_mask_iou=('iou','mean'),mask_false_positive_pixels=('false_positive_pixels_vs_reviewed','sum'),mask_false_negative_pixels=('false_negative_pixels_vs_reviewed','sum'),mask_xor_pixels=('xor_pixels','sum'),corrected_xor_pixels=('corrected_xor_pixels_vs_reviewed','sum'),corrected_mae_mean=('corrected_mae_vs_reviewed','mean'))
summary_df.to_csv(SCORE/'arm_summary.csv',index=False)
display(summary_df)

,arm,pages,mean_mask_iou,mask_false_positive_pixels,mask_false_negative_pixels,mask_xor_pixels,corrected_xor_pixels,corrected_mae_mean
0,confidence_gated_yolo,151,0.006777,77218,10658165,10735383,11888975,0.582784
1,raw_yolo,151,0.546048,3921495,2750160,6671655,8484193,0.424696
2,reviewed_comparator,151,1.000000,0,0,0,0,0.000000
3,reviewer_corrected_yolo,151,1.000000,0,0,0,0,0.000000


## 10. Optional clean-truth scoring after lock

In [11]:
# PROVENANCE: project clean-truth firewall; disk-backed bounded-memory version.
import gc
require((LOCKS/'prediction_and_output_lock.json').exists(),'Prediction/output lock missing.')
clean_map={p.stem:p for p in CLEAN_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if CLEAN_DIR.exists() else {}
truth_rows=[]; missing=[]
for page in sorted(runtime):
    cp=clean_map.get(page)
    if cp is None: missing.append(page); continue
    clean=read_gray(cp)
    for arm in ARMS:
        corrected_path=OUTPUTS/arm/page/'corrected.png'; mask_path=OUTPUTS/arm/page/'prepared_mask.png'
        require(corrected_path.exists(),f'{page}/{arm}: corrected output missing')
        require(mask_path.exists(),f'{page}/{arm}: prepared mask missing')
        corrected=read_gray(corrected_path); prepared_mask=binary(read_gray(mask_path))
        require(clean.shape==corrected.shape==prepared_mask.shape,f'{page}/{arm}: clean/output/mask shape mismatch')
        diff=np.abs(clean.astype(np.float32)-corrected.astype(np.float32)); inside=prepared_mask>0
        truth_rows.append({'page':page,'arm':arm,'whole_page_mae':float(diff.mean()),'inside_prepared_mask_mae':float(diff[inside].mean()) if np.any(inside) else np.nan,'prepared_mask_pixels':int(inside.sum())})
        del corrected,prepared_mask,diff,inside
    del clean
    gc.collect()
if truth_rows:
    truth_df=pd.DataFrame(truth_rows)
    require(len(truth_df)==(len(runtime)-len(missing))*len(ARMS),'Clean-truth scoring table is incomplete.')
    truth_df.to_csv(SCORE/'postlock_clean_truth_scores.csv',index=False)
if missing: pd.DataFrame({'page_missing_clean_counterpart':missing}).to_csv(AUDIT/'clean_truth_missing_pages.csv',index=False)
print('Truth-scored pages:',len(set(x['page'] for x in truth_rows)),'missing clean:',len(missing))

Truth-scored pages: 0 missing clean: 151


## 11. Visual audit queue

In [12]:
# PROVENANCE: V9.0A disk-backed bounded-memory visual audit.
import gc
queue=paired[paired.arm.isin(['raw_yolo','confidence_gated_yolo'])].copy()
queue['priority']=queue.corrected_xor_pixels_vs_reviewed+5*queue.false_positive_pixels_vs_reviewed+3*queue.false_negative_pixels_vs_reviewed
queue=queue.sort_values('priority',ascending=False).reset_index(drop=True)
queue.to_csv(SCORE/'visual_audit_queue.csv',index=False)
resolution_by_page=resolution.set_index('page')
for i,r in queue.head(30).iterrows():
    page=str(r.page); arm=str(r.arm)
    require(page in resolution_by_page.index,f'{page}: missing from resolution table')
    observed=read_gray(Path(resolution_by_page.loc[page,'observed_path']))
    reviewed=read_gray(MASKS/'reviewed_comparator'/f'{page}.png')>0
    candidate=read_gray(MASKS/arm/f'{page}.png')>0
    reviewed_corrected=read_gray(OUTPUTS/'reviewed_comparator'/page/'corrected.png')
    candidate_corrected=read_gray(OUTPUTS/arm/page/'corrected.png')
    require(observed.shape==reviewed.shape==candidate.shape==reviewed_corrected.shape==candidate_corrected.shape,f'{page}/{arm}: visual shape mismatch')
    overlap=reviewed&candidate; missed=reviewed&~candidate; overreach=candidate&~reviewed
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2RGB); overlay[overlap]=(0,180,0); overlay[missed]=(255,165,0); overlay[overreach]=(255,0,0)
    fig,axes=plt.subplots(1,4,figsize=(20,7)); images=[observed,overlay,reviewed_corrected,candidate_corrected]; titles=['Observed','Green overlap / orange missed / red overreach','Reviewed comparator correction',f'{arm} correction']
    for ax,image,title in zip(axes,images,titles): ax.imshow(image,cmap='gray' if image.ndim==2 else None,vmin=0 if image.ndim==2 else None,vmax=255 if image.ndim==2 else None); ax.set_title(title); ax.axis('off')
    fig.suptitle(f'{page} | {arm} | IoU={r.iou:.4f} | FP={int(r.false_positive_pixels_vs_reviewed):,} | FN={int(r.false_negative_pixels_vs_reviewed):,} | corrected XOR={int(r.corrected_xor_pixels_vs_reviewed):,}')
    plt.tight_layout(); fig.savefig(VISUAL/f'{i+1:03d}_{page}_{arm}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
    del observed,reviewed,candidate,reviewed_corrected,candidate_corrected,overlap,missed,overreach,overlay,fig,axes,images
    gc.collect()
print('Visual cases:',min(30,len(queue)))

Visual cases: 30


## 12. Fail-closed verdict

In [13]:
# PROVENANCE: V9.0A fail-closed verdict. PASS means evidence ready for review only.
expected_arms=set(ARMS); manifest_arms=set(mask_manifest.arm.astype(str).unique())
checks={
 'audit_resolver_single_version':bool(len(resolution)>0),
 'canonical_page_identity_from_notebook07':bool(resolution.page.notna().all() and ~resolution.page.duplicated().any()),
 'confidence_map_resolved_from_notebook06_audit':bool('confidence_map_path' in resolution.columns and resolution.confidence_map_path.notna().all()),
 'experimental_detector_gate_explicit':bool(EXPERIMENTAL_YOLO_GATE is not None and str(EXPERIMENTAL_YOLO_GATE_POLICY_ID).strip()),
 'all_resolved_pages_have_reviewed_masks':bool('reviewed_mask_path' in resolution.columns and resolution.reviewed_mask_path.notna().all()),
 'four_arm_roster_complete':bool(manifest_arms==expected_arms and len(mask_manifest)==len(resolution)*len(expected_arms)),
 'zero_outside_mask_drift':bool(len(output_manifest)>0 and (output_manifest.outside_mask_drift==0).all()),
 'prediction_output_lock_exists':(LOCKS/'prediction_and_output_lock.json').exists(),
 'paired_comparisons_written':(SCORE/'paired_mask_and_output_evidence.csv').exists(),
}
first_failure=next((k for k,v in checks.items() if not v),None)
verdict='PASS_V9_0A_EVIDENCE_PACKAGE_READY_FOR_REVIEW' if first_failure is None else 'FAIL_V9_0A_INVALID_RUN'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A-audit-resolved-mask-authority-transfer-streaming','verdict':verdict,'first_failed_gate':first_failure,'checks':checks,'lock_id':lock['lock_id'],'resolved_pages':len(resolution),'unresolved_review_records':len(unresolved),'unresolved_lineage_policy':UNRESOLVED_LINEAGE_POLICY,'experimental_yolo_gate':EXPERIMENTAL_YOLO_GATE,'experimental_yolo_gate_policy_id':EXPERIMENTAL_YOLO_GATE_POLICY_ID,'automatic_mask_authority_authorised':False,'production_output_authority':False,'reviewed_mask_is_comparator_not_perfect_truth':True,'reviewed_comparator_and_reviewer_corrected_yolo_are_role_labels_for_the_same_pixels':True}
atomic_json(OUT/'v9_0a_verdict.json',record); print(json.dumps(record,indent=2))
if first_failure: raise RuntimeError(f'V9.0A failed at {first_failure}')

{
  "created_utc": "2026-11-10T16:14:56.568506+00:00",
  "version": "9.0A-audit-resolved-mask-authority-transfer-streaming",
  "verdict": "PASS_V9_0A_EVIDENCE_PACKAGE_READY_FOR_REVIEW",
  "first_failed_gate": null,
  "checks": {
    "audit_resolver_single_version": true,
    "canonical_page_identity_from_notebook07": true,
    "confidence_map_resolved_from_notebook06_audit": true,
    "experimental_detector_gate_explicit": true,
    "all_resolved_pages_have_reviewed_masks": true,
    "four_arm_roster_complete": true,
    "zero_outside_mask_drift": true,
    "prediction_output_lock_exists": true,
    "paired_comparisons_written": true
  },
  "lock_id": "c5d7cc2e10ec751bfd9bd3e7708d4f65cbf83ebdef63f7230654c46f9c51348a",
  "resolved_pages": 151,
  "unresolved_review_records": 22,
  "unresolved_lineage_policy": "exclude_and_report",
  "experimental_yolo_gate": 0.9,
  "experimental_yolo_gate_policy_id": "V9.0A_EXPLORATORY_YOLO_CONFIDENCE_GE_090",
  "automatic_mask_authority_authorised": fal

## 13. Final Upload Package

In [17]:
# PROVENANCE: standard curated Final Upload Package.
required=[OUT/'v9_0a_verdict.json',AUDIT/'notebook06_draft_audit_inventory.csv',AUDIT/'notebook07_to_notebook06_resolution_attempts.csv',AUDIT/'unresolved_review_lineages.csv',AUDIT/'hitl_input_resolution.csv',AUDIT/'hitl_resolution_summary.json',AUDIT/'yolo_gate_identity.json',LOCKS/'mask_arm_manifest.csv',LOCKS/'corrected_output_manifest.csv',LOCKS/'prediction_and_output_lock.json',SCORE/'reviewer_edits.csv',SCORE/'paired_mask_and_output_evidence.csv',SCORE/'arm_summary.csv',SCORE/'visual_audit_queue.csv']
truth_path=SCORE/'postlock_clean_truth_scores.csv'
if truth_path.exists(): required.append(truth_path)
for path in required: require(path.exists(),f'Missing upload artefact: {path}')
readme=('V9.0A audit-resolved mask-authority transfer experiment.\n'
'\n'
'Notebook 07 review audits define the canonical PageXXX roster.\n'
'Notebook 06 draft evidence is resolved backwards using audited path and content identity.\n'
'Ambiguous or unresolved lineages are reported rather than guessed.\n'
'The configured higher YOLO confidence cut-off is exploratory research evidence, not production authority.\n'
'No automatic mask authority or production output is authorised.')
(UPLOAD/'README.txt').write_text(readme)
zip_path=UPLOAD/'58_V9_0A_upload_bundle.zip'
for stale in [zip_path,UPLOAD/'upload_manifest.csv',UPLOAD/'upload_file_list.txt']:
    if stale.exists(): stale.unlink()
rows=[]; visuals=sorted(VISUAL.glob('*.png'))[:30]
for src in required+visuals:
    require(src.exists(),f'Upload source disappeared: {src}')
    category='visuals' if src.suffix.lower()=='.png' else ('tables' if src.parent==SCORE else 'audit' if src.parent in [AUDIT,LOCKS] else 'root')
    dst=UPLOAD/category/src.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
    rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
upload_manifest=pd.DataFrame(rows).sort_values('relative_path').reset_index(drop=True)
require(not upload_manifest.relative_path.duplicated().any(),'Duplicate relative paths in upload package.')
upload_manifest.to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text(''.join(upload_manifest.relative_path)+'')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: archive.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
for _,row in upload_manifest.iterrows():
    packaged=UPLOAD/row.relative_path; require(packaged.exists(),f'Packaged file missing: {packaged}'); require(sha256_file(packaged)==row.sha256,f'Packaged file hash mismatch: {packaged}')
print('Upload bundle:',zip_path); print('Packaged review artefacts:',len(upload_manifest)); print('Bundle bytes:',zip_path.stat().st_size)

Upload bundle: /content/drive/MyDrive/MusicAnnotationProject/mask_authority_transfer_v9_0a/runs/20261110_151505_v9_0a_audit_resolved/upload/58_V9_0A_upload_bundle.zip
Packaged review artefacts: 44
Bundle bytes: 16683207


## Run instruction

After setting the exact historical Notebook 06 detector-confidence gate and policy identity, use **Runtime → Restart session and run all**. Do not execute from a session containing variables created by an older resolver version.